# Trabajo Practico Clase 03 - Inteligencia Artificial
### Búsqueda voraz y A* sobre un grafo dirigido
Alumno: Yamila Caviglione

## 1. REPRESENTACIÓN DEL GRAFO Y HEURÍSTICA

 Grafo dirigido. Cada estado tiene una lista de (sucesor, costo).

In [3]:
grafo = {
    "S": [("A", 2), ("B", 2)],
    "A": [("C", 2), ("D", 5)],
    "B": [("D", 2)],
    "C": [("G", 3)],
    "D": [("G", 6)],
    "G": []
}

# Heurística: estimación del costo restante hasta el objetivo G.
heuristica = {
    "S": 7,
    "A": 5,
    "B": 7,
    "C": 3,
    "D": 6,
    "G": 0
}

# Estado inicial y objetivo
inicio = "S"
objetivo = "G"

Comprobamos la representacion:


In [4]:
print("Estado inicial:", inicio)
print("Objetivo:", objetivo)

print("\nGrafo:")
for estado, sucesores in grafo.items():
    print(f"{estado} -> {sucesores}")

print("\nHeurística:")
for estado, h in heuristica.items():
    print(f"h({estado}) = {h}")

Estado inicial: S
Objetivo: G

Grafo:
S -> [('A', 2), ('B', 2)]
A -> [('C', 2), ('D', 5)]
B -> [('D', 2)]
C -> [('G', 3)]
D -> [('G', 6)]
G -> []

Heurística:
h(S) = 7
h(A) = 5
h(B) = 7
h(C) = 3
h(D) = 6
h(G) = 0


Cada nodo almacena su estado, el nodo padre, la acción realizada para llegar a él, el costo acumulado g, la heurística h y la prioridad f.

La reconstrucción del camino se realizará utilizando las referencias a los nodos padres, evitando establecer los caminos manualmente.

## 2. Estructura de datos y cola de prioridad

In [5]:
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": g + h
    }


def reconstruir_camino(nodo):
    camino = []
    actual = nodo

    while actual is not None:
        camino.append(actual["estado"])
        actual = actual["padre"]

    camino.reverse()
    return camino

Para mantener el desempate estable indicado en la consigna, cada elemento de la cola de prioridad se almacena junto con un contador de inserción. De esta forma, ante prioridades iguales, se extrae primero el nodo que fue insertado antes.

In [6]:
import heapq

In [7]:
class ColaPrioridad:
    def __init__(self):
        self.elementos = []
        self.contador = 0

    def insertar(self, nodo, prioridad):
        heapq.heappush(
            self.elementos,
            (prioridad, self.contador, nodo)
        )
        self.contador += 1

    def extraer(self):
        return heapq.heappop(self.elementos)[2]

    def esta_vacia(self):
        return len(self.elementos) == 0

    def contenido(self):
        return [
            {
                "estado": nodo["estado"],
                "prioridad": prioridad
            }
            for prioridad, _, nodo in sorted(self.elementos)
        ]

In [8]:
cola = ColaPrioridad()

nodo_a = crear_nodo("A")
nodo_b = crear_nodo("B")
nodo_c = crear_nodo("C")

cola.insertar(nodo_a, 5)
cola.insertar(nodo_b, 5)
cola.insertar(nodo_c, 3)

print(cola.extraer()["estado"])
print(cola.extraer()["estado"])
print(cola.extraer()["estado"])

C
A
B


In [9]:
def mostrar_frontera(cola):
    contenido = cola.contenido()

    if not contenido:
        return "[]"

    return [
        f"{elemento['estado']} (prioridad={elemento['prioridad']})"
        for elemento in contenido
    ]

## 3. Búsqueda de costo uniforme (UCS)

UCS utiliza g(n) como prioridad de la frontera. En cada paso se extrae el nodo con menor costo acumulado. Cuando se encuentra un camino de menor costo hacia un estado, se actualiza mejor_g y se inserta una nueva entrada en la frontera.

El objetivo se comprueba al extraer el nodo y el camino final se reconstruye utilizando las referencias a los padres.

In [10]:
def ucs(grafo, inicio, objetivo):
    frontera = ColaPrioridad()

    raiz = crear_nodo(
        estado=inicio,
        g=0,
        h=heuristica[inicio]
    )

    frontera.insertar(raiz, raiz["g"])

    mejor_g = {
        inicio: 0
    }

    generados = 1
    expandidos = 0
    frontera_maxima = 1
    reaperturas = 0
    traza = []

    while not frontera.esta_vacia():

        nodo = frontera.extraer()

        if nodo["g"] > mejor_g[nodo["estado"]]:
            continue

        if nodo["estado"] == objetivo:
            return {
                "camino": reconstruir_camino(nodo),
                "costo": nodo["g"],
                "generados": generados,
                "expandidos": expandidos,
                "frontera_maxima": frontera_maxima,
                "reaperturas": reaperturas,
                "traza": traza
            }

        expandidos += 1

        for sucesor, costo in grafo[nodo["estado"]]:

            nuevo_g = nodo["g"] + costo

            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:

                if sucesor in mejor_g:
                    reaperturas += 1

                mejor_g[sucesor] = nuevo_g

                nuevo_nodo = crear_nodo(
                    estado=sucesor,
                    padre=nodo,
                    accion=f"{nodo['estado']} -> {sucesor}",
                    g=nuevo_g,
                    h=heuristica[sucesor]
                )

                frontera.insertar(
                    nuevo_nodo,
                    nuevo_nodo["g"]
                )

                generados += 1

        frontera_maxima = max(
            frontera_maxima,
            len(frontera.elementos)
        )

        traza.append({
            "extraido": nodo["estado"],
            "frontera": mostrar_frontera(frontera)
        })

    return {
        "camino": None,
        "costo": None,
        "generados": generados,
        "expandidos": expandidos,
        "frontera_maxima": frontera_maxima,
        "reaperturas": reaperturas,
        "traza": traza
    }

In [11]:
resultado_ucs = ucs(
    grafo,
    inicio,
    objetivo
)

print("Camino:", " -> ".join(resultado_ucs["camino"]))
print("Costo:", resultado_ucs["costo"])
print("Estados generados:", resultado_ucs["generados"])
print("Estados expandidos:", resultado_ucs["expandidos"])
print("Frontera máxima:", resultado_ucs["frontera_maxima"])
print("Reaperturas:", resultado_ucs["reaperturas"])

Camino: S -> A -> C -> G
Costo: 7
Estados generados: 7
Estados expandidos: 5
Frontera máxima: 3
Reaperturas: 1


In [12]:
print("TRAZA DE UCS")

for i, paso in enumerate(resultado_ucs["traza"], start=1):
    print(f"\nExpansión {i}")
    print("Estado extraído:", paso["extraido"])
    print("Frontera:", paso["frontera"])

TRAZA DE UCS

Expansión 1
Estado extraído: S
Frontera: ['A (prioridad=2)', 'B (prioridad=2)']

Expansión 2
Estado extraído: A
Frontera: ['B (prioridad=2)', 'C (prioridad=4)', 'D (prioridad=7)']

Expansión 3
Estado extraído: B
Frontera: ['C (prioridad=4)', 'D (prioridad=4)', 'D (prioridad=7)']

Expansión 4
Estado extraído: C
Frontera: ['D (prioridad=4)', 'D (prioridad=7)', 'G (prioridad=7)']

Expansión 5
Estado extraído: D
Frontera: ['D (prioridad=7)', 'G (prioridad=7)']


## 4. Busqueda Voraz


**Búsqueda voraz por el mejor primero**

La búsqueda voraz utiliza h(n) como prioridad de la frontera. En cada paso selecciona el estado que, según la heurística, parece estar más cerca del objetivo.

A diferencia de UCS, la prioridad no tiene en cuenta el costo acumulado del camino recorrido. El objetivo se comprueba al extraer el nodo y el camino se reconstruye mediante las referencias a los padres.

In [13]:
def voraz(grafo, inicio, objetivo):
    frontera = ColaPrioridad()

    raiz = crear_nodo(
        estado=inicio,
        g=0,
        h=heuristica[inicio]
    )

    frontera.insertar(raiz, raiz["h"])

    mejor_g = {
        inicio: 0
    }

    generados = 1
    expandidos = 0
    frontera_maxima = 1
    reaperturas = 0
    traza = []

    while not frontera.esta_vacia():

        nodo = frontera.extraer()

        if nodo["g"] > mejor_g[nodo["estado"]]:
            continue

        if nodo["estado"] == objetivo:
            return {
                "camino": reconstruir_camino(nodo),
                "costo": nodo["g"],
                "generados": generados,
                "expandidos": expandidos,
                "frontera_maxima": frontera_maxima,
                "reaperturas": reaperturas,
                "traza": traza
            }

        expandidos += 1

        for sucesor, costo in grafo[nodo["estado"]]:

            nuevo_g = nodo["g"] + costo

            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:

                if sucesor in mejor_g:
                    reaperturas += 1

                mejor_g[sucesor] = nuevo_g

                nuevo_nodo = crear_nodo(
                    estado=sucesor,
                    padre=nodo,
                    accion=f"{nodo['estado']} -> {sucesor}",
                    g=nuevo_g,
                    h=heuristica[sucesor]
                )

                frontera.insertar(
                    nuevo_nodo,
                    nuevo_nodo["h"]
                )

                generados += 1

        frontera_maxima = max(
            frontera_maxima,
            len(frontera.elementos)
        )

        traza.append({
            "extraido": nodo["estado"],
            "frontera": mostrar_frontera(frontera)
        })

    return {
        "camino": None,
        "costo": None,
        "generados": generados,
        "expandidos": expandidos,
        "frontera_maxima": frontera_maxima,
        "reaperturas": reaperturas,
        "traza": traza
    }

In [14]:
resultado_voraz = voraz(
    grafo,
    inicio,
    objetivo
)

print("Camino:", " -> ".join(resultado_voraz["camino"]))
print("Costo:", resultado_voraz["costo"])
print("Estados generados:", resultado_voraz["generados"])
print("Estados expandidos:", resultado_voraz["expandidos"])
print("Frontera máxima:", resultado_voraz["frontera_maxima"])
print("Reaperturas:", resultado_voraz["reaperturas"])

Camino: S -> A -> C -> G
Costo: 7
Estados generados: 6
Estados expandidos: 3
Frontera máxima: 3
Reaperturas: 0


In [15]:
print("TRAZA DE VORAZ")

for i, paso in enumerate(resultado_voraz["traza"], start=1):
    print(f"\nExpansión {i}")
    print("Estado extraído:", paso["extraido"])
    print("Frontera:", paso["frontera"])

TRAZA DE VORAZ

Expansión 1
Estado extraído: S
Frontera: ['A (prioridad=5)', 'B (prioridad=7)']

Expansión 2
Estado extraído: A
Frontera: ['C (prioridad=3)', 'D (prioridad=6)', 'B (prioridad=7)']

Expansión 3
Estado extraído: C
Frontera: ['G (prioridad=0)', 'D (prioridad=6)', 'B (prioridad=7)']


## 5. Busqueda A*

A* utiliza como prioridad la función f(n) = g(n) + h(n). De esta manera combina el costo acumulado del camino recorrido con la estimación del costo restante hasta el objetivo.

En cada extracción se descartan las entradas obsoletas, se comprueba si el estado es el objetivo y luego se relajan sus sucesores. Cuando se encuentra un costo acumulado menor para un estado ya conocido, se actualiza mejor_g y se inserta una nueva entrada en la frontera.

In [16]:
def a_estrella(grafo, inicio, objetivo):
    frontera = ColaPrioridad()

    raiz = crear_nodo(
        estado=inicio,
        g=0,
        h=heuristica[inicio]
    )

    frontera.insertar(
        raiz,
        raiz["f"]
    )

    mejor_g = {
        inicio: 0
    }

    generados = 1
    expandidos = 0
    frontera_maxima = 1
    reaperturas = 0
    traza = []

    while not frontera.esta_vacia():

        nodo = frontera.extraer()

        if nodo["g"] > mejor_g[nodo["estado"]]:
            continue

        if nodo["estado"] == objetivo:
            return {
                "camino": reconstruir_camino(nodo),
                "costo": nodo["g"],
                "generados": generados,
                "expandidos": expandidos,
                "frontera_maxima": frontera_maxima,
                "reaperturas": reaperturas,
                "traza": traza
            }

        expandidos += 1

        for sucesor, costo in grafo[nodo["estado"]]:

            nuevo_g = nodo["g"] + costo

            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:

                if sucesor in mejor_g:
                    reaperturas += 1

                mejor_g[sucesor] = nuevo_g

                nuevo_nodo = crear_nodo(
                    estado=sucesor,
                    padre=nodo,
                    accion=f"{nodo['estado']} -> {sucesor}",
                    g=nuevo_g,
                    h=heuristica[sucesor]
                )

                frontera.insertar(
                    nuevo_nodo,
                    nuevo_nodo["f"]
                )

                generados += 1

        frontera_maxima = max(
            frontera_maxima,
            len(frontera.elementos)
        )

        traza.append({
            "extraido": nodo["estado"],
            "frontera": mostrar_frontera(frontera)
        })

    return {
        "camino": None,
        "costo": None,
        "generados": generados,
        "expandidos": expandidos,
        "frontera_maxima": frontera_maxima,
        "reaperturas": reaperturas,
        "traza": traza
    }

In [17]:
resultado_a_estrella = a_estrella(
    grafo,
    inicio,
    objetivo
)

print("Camino:", " -> ".join(resultado_a_estrella["camino"]))
print("Costo:", resultado_a_estrella["costo"])
print("Estados generados:", resultado_a_estrella["generados"])
print("Estados expandidos:", resultado_a_estrella["expandidos"])
print("Frontera máxima:", resultado_a_estrella["frontera_maxima"])
print("Reaperturas:", resultado_a_estrella["reaperturas"])

Camino: S -> A -> C -> G
Costo: 7
Estados generados: 6
Estados expandidos: 3
Frontera máxima: 3
Reaperturas: 0


In [18]:
print("TRAZA DE A*")

for i, paso in enumerate(resultado_a_estrella["traza"], start=1):
    print(f"\nExpansión {i}")
    print("Estado extraído:", paso["extraido"])
    print("Frontera:", paso["frontera"])

TRAZA DE A*

Expansión 1
Estado extraído: S
Frontera: ['A (prioridad=7)', 'B (prioridad=9)']

Expansión 2
Estado extraído: A
Frontera: ['C (prioridad=7)', 'B (prioridad=9)', 'D (prioridad=13)']

Expansión 3
Estado extraído: C
Frontera: ['G (prioridad=7)', 'B (prioridad=9)', 'D (prioridad=13)']


## 6. Comparación de resultados

| Resultado                     | UCS           | Voraz         | A*            |
| ----------------------------- | ------------- | ------------- | ------------- |
| Camino                        | S → A → C → G | S → A → C → G | S → A → C → G |
| Costo                         | 7             | 7             | 7             |
| Prioridad                     | g(n)          | h(n)          | g(n) + h(n)   |
| Expandidos antes de extraer G | 5             | 3             | 3             |

Los tres algoritmos encuentran el mismo camino de costo 7 en este grafo. Sin embargo, utilizan diferentes criterios para ordenar la frontera. UCS utiliza el costo acumulado `g(n)`, Voraz utiliza solamente la heurística `h(n)` y A* combina ambos mediante `g(n) + h(n)`. En esta instancia, Voraz y A* expanden menos estados que UCS, aunque los tres obtienen el mismo costo de solución.



## 7. Respuesta a preguntas:


### 1. ¿Por qué voraz y A* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?

Voraz y A* coinciden porque la heurística utilizada orienta a ambos hacia la misma rama. Desde S, Voraz prefiere A porque h(A)=5 es menor que h(B)=7. A* también elige A, ya que f(A)=2+5=7, mientras que f(B)=2+7=9. Luego, desde A, ambos continúan por C, y finalmente llegan a G. Además, la heurística es admisible y consistente en este grafo, por lo que no sobreestima el costo restante y cumple las condiciones analizadas para A*. Por eso, en esta instancia ambos encuentran S → A → C → G, con costo 7.

### 2. ¿Garantiza voraz devolver el camino de menor costo en general? Justificá con la propiedad de su prioridad (no con este ejemplo).

No. Voraz no garantiza encontrar el camino de menor costo porque su prioridad es solamente h(n). Esto significa que tiene en cuenta qué tan cerca parece estar un estado del objetivo, pero ignora cuánto costó llegar hasta ese estado. Por eso puede elegir una ruta que parece prometedora según la heurística pero que en realidad tiene un costo acumulado muy alto. La clase muestra que Voraz no es óptimo en general, incluso cuando la heurística es admisible.

### 3. ¿Qué ocurre si se usa h = 0 en A*? ¿Con qué algoritmo coincide entonces?

Si se utiliza h(n)=0 para todos los estados, la función de prioridad de A* queda f(n)=g(n)+0, por lo que f(n)=g(n). Entonces A* utiliza la misma prioridad que UCS y coincide con la búsqueda de costo uniforme.

### 4. ¿Hubo reaperturas en UCS? ¿Y en A* y voraz? ¿Por qué?

En UCS hubo una reapertura. Primero se generó D con costo 7 mediante el camino S → A → D. Después, al expandir B, se encontró otro camino hacia D con costo 4 mediante S → B → D. Como se obtuvo un g menor para un estado que ya era conocido, se actualizó mejor_g[D] y se contó una reapertura.

En Voraz y A* no hubo reaperturas en esta ejecución, porque no se encontró posteriormente un camino con un g menor para un estado que ya había sido registrado.

### 5. ¿"Expandir menos estados" significa "camino más barato"? Relacionalo con lo que muestran UCS y voraz aquí.

No. Expandir menos estados y obtener un camino más barato son medidas diferentes. En este caso, Voraz expandió 3 estados y UCS expandió 5, pero ambos encontraron un camino de costo 7. Sin embargo, esto no significa que Voraz siempre encuentre caminos de ese costo. Como utiliza solamente h(n), puede expandir pocos estados y terminar encontrando una solución más cara. La cantidad de expansiones mide cuánto exploró el algoritmo, mientras que el costo mide la calidad de la solución encontrada.